# LLDT end-to-end latency analysis

This notebook is the plotting template used to analyze the end-to-end latency samples produced by the benchmark harness.

The measured path is:

```text
Producer
  -> SHM
  -> LLDT Sender
  -> DPDK / Ethernet / IPv4 / UDP
  -> LLDT Receiver
  -> SHM
  -> Consumer
```

The Consumer writes one `latency.csv` per benchmark run. The Python cells below expect the active sample file at:

```text
data/latency.csv
```

For the submitted benchmark sweep, the raw CSV files were not committed because the large runs produce very large measurement files. Instead, the resulting distribution and tail plots are committed under `docs/benchmarks/`, and the complete per-run counters and percentile summaries are recorded in:

- [E2E_RESULTS.md](E2E_RESULTS.md)
- [E2E_OPTIMIZED_RESULTS.md](E2E_OPTIMIZED_RESULTS.md)

For batch plot generation, downloaded run files can be placed at:

```text
benchmark-results/<RUN-ID>/latency.csv
```

and processed with:

```bash
./scripts/render_latency_plots.sh
```

The script feeds each CSV through this notebook and writes the generated PNGs to `docs/benchmarks/`.


In [ ]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter
from bokeh.plotting import figure

output_notebook()

BLUE = "#2a78d6"

ns = pd.read_csv("data/latency.csv")["latency_ns"].to_numpy()
ns.size, ns.min(), int(ns.mean()), ns.max()

## Distribution

Clipped at p99.9 for display purposes

In [ ]:
clip = np.percentile(ns, 99.9)
counts, edges = np.histogram(ns[ns <= clip], bins=60)
src = ColumnDataSource(dict(count=counts, left=edges[:-1], right=edges[1:],
                            mid=(edges[:-1] + edges[1:]) / 2))

p = figure(height=320, sizing_mode="stretch_width",
           title=f"Delivery latency, clipped at p99.9 = {clip:,.0f} ns",
           x_axis_label="latency (ns)", y_axis_label="messages",
           tools="pan,box_zoom,reset,save")
p.quad(source=src, bottom=0, top="count", left="left", right="right",
       fill_color=BLUE, line_color="white")
p.y_range.start = 0
p.xaxis.formatter = p.yaxis.formatter = NumeralTickFormatter(format="0,0")
p.add_tools(HoverTool(tooltips=[("latency", "@mid{0,0} ns"),
                                ("messages", "@count{0,0}")], mode="vline"))
show(p)

## Tail

Log axis on both scales — a mean hides the tail entirely.

In [ ]:
# Plotted against "distance from 100th percentile" on a log axis, so the tail gets
# real width instead of being squeezed into the last few pixels.
# Stops at p99.99: beyond that is a handful of samples out of 200k, not a stable figure.
pcts = np.array([50, 75, 90, 99, 99.9, 99.99])
curve = ColumnDataSource(dict(x=100 - pcts, ns=np.percentile(ns, pcts),
                              pct=[f"p{v:g}" for v in pcts]))

q = figure(height=320, sizing_mode="stretch_width",
           x_axis_type="log", y_axis_type="log",
           x_range=(70, 0.0025),  # reversed: p50 on the left, p99.99 on the right
           title="Latency tail",
           x_axis_label="percentile", y_axis_label="latency (ns, log)",
           tools="pan,box_zoom,reset,save")
q.line(source=curve, x="x", y="ns", color=BLUE, width=2)
q.scatter(source=curve, x="x", y="ns", color=BLUE, size=9,
          line_color="white", line_width=2)
q.xaxis.ticker = list(100 - pcts)
q.xaxis.major_label_overrides = {100 - v: f"p{v:g}" for v in pcts}
q.yaxis.ticker = [100, 200, 500, 1000, 2000, 5000]
q.yaxis.formatter = NumeralTickFormatter(format="0,0")
q.add_tools(HoverTool(tooltips=[("percentile", "@pct"), ("latency", "@ns{0,0} ns")]))
show(q)

pd.DataFrame([{f"p{v:g}": int(x) for v, x in zip(pcts, np.percentile(ns, pcts))}])

# Submitted benchmark analysis

The main benchmark was a four-way ablation:

| Message profile | Source-frame batching |
|---|---|
| Raw | OFF |
| Raw | ON |
| Compact | OFF |
| Compact | ON |

The full numerical results, Sender/Receiver counters, clock snapshots, and plots for every submitted run are available in [E2E_RESULTS.md](E2E_RESULTS.md).

## Packet-rate saturation without effective batching

With Raw messages and batching disabled, the path was clean at `20k msg/s` but already saturated at `50k msg/s`.

At `50k msg/s`:

- Sender reported local TX-unsent packets;
- Receiver missing Data sequences tracked those unsent packets;
- source-SHM lapping also appeared as the Sender fell behind;
- Consumer drop rate reached `3.7719%`;
- p50 latency increased to approximately `290 us`.

Tail plot:

![Raw + batching OFF, 50k msg/s](docs/benchmarks/20260826-141219092-r50000-n1000000-tail.png)

This indicates that the initial limitation was packet-rate / Sender-TX pressure rather than network bandwidth.

## Batching improves Raw throughput, but message size limits packing

With Raw + batching enabled, the transport remained clean through `100k msg/s`.

At requested `400k msg/s`, however, the configuration clearly saturated:

- average batching was approximately `2.69 frames/packet`;
- Consumer drop rate reached `23.27%`;
- p50 latency was approximately `371 us`.

Tail plot:

![Raw + batching ON, 400k msg/s](docs/benchmarks/20260826-145650501-r400000-n4000000-tail.png)

The original Raw records are large enough that packet-count reduction from batching is limited.

## Compact + batching

Compact + batching was the strongest tested configuration.

At requested `200k msg/s`:

- Consumer drop rate: `0%`
- p50: `27.2 us`
- p99: `50.6 us`
- p99.9: `67.5 us`
- p99.99: `81.9 us`
- average batching: approximately `5.75 frames/packet`

Distribution:

![Compact + batching ON, 200k msg/s — distribution](docs/benchmarks/20260826-152951278-r200000-n10000000-distribution.png)

Tail:

![Compact + batching ON, 200k msg/s — tail](docs/benchmarks/20260826-152951278-r200000-n10000000-tail.png)

At requested `400k msg/s`, batching increased to approximately `6.65 frames/packet`, while the measured path still showed:

- no Sender source gaps;
- no TX-unsent packets;
- no Receiver missing Data packets;
- no Consumer drops.

Tail:

![Compact + batching ON, 400k msg/s](docs/benchmarks/20260826-153223834-r400000-n4000000-tail.png)

## Load ceiling

The Compact mixed Producer itself was measured separately with pacing disabled:

```text
16,000,000 messages / 23.10 s ~= 693,000 msg/s
```

Therefore approximately `700k msg/s` was the practical source-generation ceiling of the benchmark machine.

The requested `800k` and `1.6M msg/s` runs should not be interpreted as actually offering those rates. Compact + batching did not reach transport saturation before the Producer became the limiting component.

## TX-burst experiment

A later bounded opportunistic multi-mbuf TX-burst experiment improved several latency percentiles, but the measured number of Data packets per TX call remained close to `1`. Source-frame packing also changed substantially, and the requested `200k msg/s` run introduced a small `0.0038%` source-side loss.

The experiment was therefore rejected rather than merged into the final Sender.

Its complete measurements are retained in [E2E_OPTIMIZED_RESULTS.md](E2E_OPTIMIZED_RESULTS.md).

## Conclusion

The benchmark evidence indicates that reducing **packet rate** was the most important optimization for this implementation.

Compact encoding alone did not solve the packet-rate ceiling when every source message still generated its own UDP packet. Combining the Compact source representation with opportunistic source-frame batching reduced the number of Data packets sufficiently to remove the observed Sender-TX saturation across the load range the Producer could generate.

The submitted default is therefore:

```text
Compact + source-frame batching ON
```

with the original single-packet TX submission path.
